# Distributed Topological Areas

Distributed applications (e.g. a controller for one section of a feeder) only need the part of the network they manage. A distributed `FeederModel` splits a feeder into a hierarchy of areas, each with its own typed property graph:

* `FeederArea`: the whole feeder, from the substation to the first switching devices

* `SwitchArea`: the network between switching devices, inside the feeder area

* `SecondaryArea`: the customer-side network behind a distribution transformer, inside a switch area

In the `cimhub_2023` profile these are `SubSchedulingArea` classes. Each area is a `DistributedArea` (a `GraphModel`) whose `container` is the area object, whose `graph` holds only the objects of that area, and whose `distributed_areas` list holds the areas below it. `FeederModel.distributed_areas` holds the feeder area.

A distributed `FeederModel` is created with `distributed=True` in one of two ways:

1. From a database that already contains the areas (`Feeder.FeederArea`, `FeederArea.SwitchAreas`, `SwitchArea.SecondaryAreas`), as created by the `create_distributed_feeder()` method of the CIM-Graph-Topology-Processor library. Each area is then loaded with the database queries for its equipment, nodes, terminals and measurements.

2. From a topology message (`topology_message=`) of the GridAPPS-D Topology Processor, which lists the equipment of each area.

The XML and JSON-LD file parsers do not support distributed models, as they cannot query an area. The example below therefore uses the `RDFlibConnection` (`pip install cim-graph[rdflib]`), which runs SPARQL queries on a file in memory.

## Building Areas from a Topology Message

Import the profile and read the IEEE 13 node feeder once with `XMLFile`, to find the equipment for the message:

In [1]:
import os
os.environ['CIMG_CIM_PROFILE'] = 'cimhub_2023'
import cimprofiles.cim17.cimhub_2023 as cim

from cimgraph.databases import RDFlibConnection, XMLFile
from cimgraph.models import FeederModel

model_file = '../../tests/test_models/ieee13.xml'
feeder = cim.Feeder(mRID='49AD8E07-3BF9-A4E2-CB8F-C3722F837B62')
centralized = FeederModel(container=feeder, connection=XMLFile(filename=model_file))

A topology message is normally produced by the topology processor. For this example we write a small one by hand: the switch area is the part of the feeder downstream of switch `671692` (buses `692` and `675`), and the feeder area holds the rest. The switch's terminals are the boundary of the switch area. In each area, the controllable devices (switches, capacitors and DERs) are listed as addressable equipment.

In [2]:
from uuid import uuid4

switch = centralized.find_by_attribute(cim.LoadBreakSwitch, 'name', '671692')[0]
switch_area_buses = {'692', '675'}

feeder_area_equipment = []
switch_area_equipment = []
for cim_class in centralized.graph:
    if not issubclass(cim_class, cim.ConductingEquipment):
        continue
    for equipment in centralized.list_by_class(cim_class):
        buses = {terminal.ConnectivityNode.name for terminal in equipment.Terminals}
        if equipment is not switch and buses <= switch_area_buses:
            switch_area_equipment.append(equipment)
        else:
            feeder_area_equipment.append(equipment)

print(len(feeder_area_equipment), 'feeder area equipment')
print('switch area equipment:', [equipment.name for equipment in switch_area_equipment])

38 feeder area equipment
switch area equipment: ['cap1', '692675', '675c', '692', '675b', '675a']


In [3]:
CONTROLLABLE = (cim.Switch, cim.ShuntCompensator, cim.PowerElectronicsConnection)

def reference(obj):
    return {'@id': obj.uri(), '@type': obj.__class__.__name__}

def area_message(area_class, equipment, boundary_terminals):
    return {
        '@id': str(uuid4()),
        '@type': area_class.__name__,
        'BoundaryTerminals': [reference(terminal) for terminal in boundary_terminals],
        'AddressableEquipment': [reference(eq) for eq in equipment if isinstance(eq, CONTROLLABLE)],
        'UnaddressableEquipment': [reference(eq) for eq in equipment if not isinstance(eq, CONTROLLABLE)],
        'Measurements': [],
    }

switch_area = area_message(cim.SwitchArea, switch_area_equipment, switch.Terminals)
switch_area['SecondaryAreas'] = []
feeder_area = area_message(cim.FeederArea, feeder_area_equipment, [])
feeder_area['SwitchAreas'] = [switch_area]
topology_message = {**reference(feeder), 'FeederArea': feeder_area}

Create the distributed model from the message:

In [4]:
connection = RDFlibConnection(filename=model_file)
network = FeederModel(container=None, connection=connection, distributed=True,
                      topology_message=topology_message)

The model's own graph holds only the feeder and its feeder area. The objects of each area are in that area's graph:

In [5]:
def print_area(area, indent=''):
    counts = {cim_class.__name__: len(objects) for cim_class, objects in area.graph.items()}
    print(indent + area.container.__class__.__name__, counts)
    for lower_area in area.distributed_areas:
        print_area(lower_area, indent + '    ')

print({cim_class.__name__: len(objects) for cim_class, objects in network.graph.items()})
for area in network.distributed_areas:
    print_area(area)

{'Feeder': 1, 'FeederArea': 1}
FeederArea {'FeederArea': 1, 'PowerElectronicsConnection': 5, 'LinearShuntCompensator': 1, 'LoadBreakSwitch': 2, 'Breaker': 1, 'Fuse': 1, 'Recloser': 1, 'EnergySource': 1, 'PowerTransformer': 4, 'ACLineSegment': 10, 'EnergyConsumer': 12, 'SwitchArea': 1}
    SwitchArea {'SwitchArea': 1, 'Terminal': 2, 'LinearShuntCompensator': 1, 'ACLineSegment': 1, 'EnergyConsumer': 4}


As with a centralized database model, the area objects start with only their identifiers. `get_all_edges()` on an area reads the attributes of a class for that area only:

In [6]:
switch_area_model = network.distributed_areas[0].distributed_areas[0]
switch_area_model.get_all_edges(cim.EnergyConsumer)

for load in switch_area_model.list_by_class(cim.EnergyConsumer):
    print(load.name, load.p.to('kW'), 'kW', load.q.to('kVAr'), 'kVAr')

675c 290.0 kW 212.0 kVAr
692 170.0 kW 151.0 kVAr
675b 68.0 kW 60.0 kVAr
675a 485.0 kW 190.0 kVAr


## Building Areas from a Database

When the database already contains the area objects, no message is needed. The example below needs a database with a distributed feeder model, so it is not run here:

```python
network = FeederModel(container=feeder, connection=database, distributed=True)
feeder_area_model = network.distributed_areas[0]
for switch_area_model in feeder_area_model.distributed_areas:
    print(switch_area_model.container.name)
```

If the feeder has no `FeederArea`, a `ValueError` asks for the model to be rebuilt with `create_distributed_feeder()`.